# Validação: reflexões, cobertura e escolha do threshold

Este notebook lê a etapa parcial (baseline + autorreflexão) ou a etapa final (também reflexões do professor). Não executa modelos. O GPT-5.4 não é estudante no novo experimento.

1. Configuração e disponibilidade
2. Auditoria das falhas
3. Resultados sem corte de similaridade
4. Thresholds: acurácia, ganho nas mesmas questões e cobertura
5. Faixas disjuntas de similaridade
6. Seleção de thresholds na validação e exportações

A versão anterior foi preservada em `validation_accuracy_by_similarity_legacy.ipynb`.

In [ ]:
USE_BASELINE_ON_FAILURE = False
EXPERIMENT_ID = None  # None = run de validação ativa; ou informe o ID explicitamente
PHASE = "auto"  # auto: final se disponível, senão parcial; self: parcial preservado; full: final
THRESHOLDS = [-1.0] + [i / 20 for i in range(21)]
MIN_N = 30
MIN_COVERAGE = 0.10
AUTO_YLIM = True
CLEAN_EXCLUDE_FLAGS = [
    "length_exhausted", "empty_exhausted", "context_exceeded", "content_filter",
    "partial_think", "reflection_clipped_for_context", "embedding_truncated",
]
# thinking_removed e judge_fallback não são falhas por si só.

## 1. Dados e denominadores

Com **True**, falhas e casos abaixo do threshold usam o baseline da mesma questão; se ele também falhar, contamos erro. Com **False**, esses casos saem do denominador e condições podem ter amostras diferentes. Uma resposta válida errada continua como erro.

A acurácia de cada condição é descritiva de seu conjunto. O **ganho** usa o baseline nas **mesmas questões** daquela condição, evitando comparar subconjuntos diferentes. Baselines não resolvidos nesses casos contam como erro e são explicitados em `n_baseline_unanswered`.

Os filtros adicionais só afetam a vista `filtrado`. Não criamos linhas de teacher ausente nem interpretamos a etapa ainda não rodada como falha. Similaridade igual ao threshold é aceita. Os modelos, limites, pares e prompts são os congelados no manifesto.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_experiment.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from validation_ops import validation_id
from rmcq.test_analysis import load_run, GROUP
from rmcq.validation_analysis import study, export_study, plot_study

RUN_ID = EXPERIMENT_ID or validation_id()
rows, sources = load_run(ROOT, RUN_ID, "validation", phase=PHASE)
if rows.empty:
    raise FileNotFoundError("Sem resultados: execute self-eval ou restaure o pacote self-eval/finish.")
manifest_path = ROOT / "experiment_exchange" / RUN_ID / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
print("Run:", RUN_ID, "| fase:", PHASE, "| professor:", manifest.get("teacher_role", "histórico"))
print("Fontes:", sources)
display(rows.groupby(GROUP).size().rename("n_original").reset_index())
CONFIG = dict(run_id=RUN_ID, fallback=USE_BASELINE_ON_FAILURE, phase=PHASE,
              thresholds=THRESHOLDS, min_n=MIN_N, min_coverage=MIN_COVERAGE,
              exclude_flags=CLEAN_EXCLUDE_FLAGS, sources=sources, auto_ylim=AUTO_YLIM)
CONFIG_ID = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:10]
OUT = ROOT / "data/results/reflection_top1" / RUN_ID / "analysis" / f"validation_notebook_{CONFIG_ID}"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "config.json").write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")

def display_metrics(table):
    columns = [c for c in table if "accuracy" in c or c in ("delta_vs_baseline", "coverage", "fallback_rate")]
    display(table.style.format({c:"{:.2%}" for c in columns}, na_rep="—"))

## 2. Auditoria das falhas

As marcações se sobrepõem; suas contagens não devem ser somadas como questões distintas. `judge_fallback` é o uso do avaliador local para interpretar a resposta, não a substituição pela resposta baseline. Os limites de geração e os blocos de thinking são auditados como no teste anterior.

In [ ]:
audit = rows[[*GROUP, "audit_flags"]].explode("audit_flags").dropna(subset=["audit_flags"])
audit = audit.groupby([*GROUP, "audit_flags"]).size().rename("n").reset_index()
audit.to_csv(OUT / "audit_flags.csv", index=False)
display(audit.groupby("audit_flags")["n"].sum().sort_values(ascending=False).to_frame())
display(rows.groupby(GROUP).agg(n=("failure","size"), falhas=("failure","sum"),
                               auditoria_detalhada=("audit_detail_available","mean")))
views = study(rows, fallback=USE_BASELINE_ON_FAILURE, thresholds=THRESHOLDS,
              clean_flags=CLEAN_EXCLUDE_FLAGS, min_n=MIN_N, min_coverage=MIN_COVERAGE)
export_study(views, OUT)

## 3. Resultados sem corte de similaridade

`n_total`: universo original; `n`: questões utilizadas; `n_reflection`: respostas com reflexão; `n_fallback`: respostas substituídas pelo baseline. `n_corrected` e `n_harmed` contam mudanças de erro para acerto e de acerto para erro, respectivamente, nas mesmas questões.

In [ ]:
for view, tables in views.items():
    print(view)
    display_metrics(tables["accuracy"])

## 4. Thresholds cumulativos

Cada threshold t inclui similaridades **≥ t**. As três colunas mostram acurácia, ganho sobre o baseline nas mesmas questões e cobertura. A linha baseline de acurácia é a referência geral; use a coluna de ganho para comparar corretamente os subconjuntos. As escalas de acurácia/ganho são compartilhadas entre vistas completa e filtrada por modelo/dataset.

As tabelas de médias incluem **macro** (peso igual por dataset) e **micro** (peso por questão). Se um dataset ficar sem amostras, a macro fica indefinida. As curvas são exploratórias; valores extremos com poucas questões exigem atenção à cobertura.

In [ ]:
plot_study(views, OUT, auto_ylim=AUTO_YLIM)
for view, tables in views.items():
    print("Médias macro/micro:", view)
    display_metrics(tables["averages"].loc[tables["averages"].threshold.isin([-1, .7, .8, .85, .9, .95])])

## 5. Faixas disjuntas de similaridade

Estas faixas complementam os thresholds: `[0.90,0.95)` contém apenas questões nesse intervalo. Consulte `matched_baseline_accuracy` e `delta_vs_baseline`: uma acurácia menor pode coexistir com um ganho maior sobre o baseline.

In [ ]:
MODEL_FILTER = None  # ex.: "ministral-3-8b"
DATASET_FILTER = None  # ex.: "logiqa2"
for view, tables in views.items():
    table = tables["bands"]
    if MODEL_FILTER:
        table = table.loc[table.model.eq(MODEL_FILTER)]
    if DATASET_FILTER:
        table = table.loc[table.dataset.eq(DATASET_FILTER)]
    print(view)
    display_metrics(table)

## 6. Thresholds escolhidos na validação

Seleção por modelo × dataset × condição: maior acurácia na grade, com `n >= MIN_N` e `coverage >= MIN_COVERAGE`; empates usam o menor threshold. Uma condição sem amostras elegíveis não recebe threshold. Não usamos resultados de teste nesta seleção.

Com fallback desligado, essa escolha otimiza a acurácia entre os casos aceitos e deve ser lida junto à cobertura. O valor encontrado ainda precisa ser avaliado no teste com o mesmo modelo, limites e política; não é um ótimo universal nem evidência de significância estatística. Alterar a política de fallback pode mudar a escolha.

In [ ]:
for view, tables in views.items():
    print("Thresholds selecionados:", view)
    display_metrics(tables["selected"])
print("Tabelas, configuração e figuras:", OUT)